# Coach Financiero Generativo - Prototipo

Notebook de prueba para validar la arquitectura del coach financiero usando Groq API con modelos open-source.

**Objetivo**: Probar varios LLMs con datos financieros reales y comparar calidad de respuestas.

In [1]:
# Celda 1 - Setup
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from pathlib import Path

# Cargar variables de entorno
load_dotenv()

# Verificar API key
api_key = os.getenv("GROQ_API_KEY")
if not api_key or api_key == "your_groq_api_key_here":
    raise ValueError(
        "Configura tu GROQ_API_KEY en el archivo .env\n"
        "Obtener gratis en: https://console.groq.com/keys"
    )

client = Groq(api_key=api_key)
print("Groq client inicializado correctamente")

Groq client inicializado correctamente


In [2]:
# Celda 2 - Carga y parseo de datos financieros
CSV_PATH = Path("../../P2_AP-IA/data/raw/db_mod_descript.csv")

df = pd.read_csv(CSV_PATH)

# Parsear importes: "10,00\u20ac" -> 10.00
df["Amount_clean"] = (
    df["Amount"]
    .str.replace("\u20ac", "", regex=False)
    .str.replace(".", "", regex=False)   # separador miles
    .str.replace(",", ".", regex=False)  # decimal
    .astype(float)
)

# Parsear fechas
df["Date_parsed"] = pd.to_datetime(df["Date"], dayfirst=True)
df["Year"] = df["Date_parsed"].dt.year
df["Month"] = df["Date_parsed"].dt.month
df["YearMonth"] = df["Date_parsed"].dt.to_period("M")

# Ordenar por fecha descendente
df = df.sort_values("Date_parsed", ascending=False).reset_index(drop=True)

print(f"Transacciones cargadas: {len(df)}")
print(f"Rango: {df['Date_parsed'].min().strftime('%d/%m/%Y')} - {df['Date_parsed'].max().strftime('%d/%m/%Y')}")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nAreas: {df['Area'].unique().tolist()}")
print(f"Tipos: {df['Type'].unique().tolist()}")
df.head(10)

Transacciones cargadas: 885
Rango: 01/09/2021 - 28/02/2026

Columnas: ['Description', 'Date', 'Amount', 'Area', 'Type', 'Amount_clean', 'Date_parsed', 'Year', 'Month', 'YearMonth']

Areas: ['Leisure', 'Salary', 'Leisure, Vacations', 'Food', 'Invoice', 'Investment', 'Deposit', 'Food, Vacations', 'Invoice, Vacations']
Tipos: ['Expenses', 'Income']


,Description,Date,Amount,Area,Type,Amount_clean,Date_parsed,Year,Month,YearMonth
0,Cena y copas en Cinesa con el grupo de amigos,28/02/2026,"10,00€",Leisure,Expenses,10.00,2026-02-28,2026,2,2026-02
1,Transferencia de sueldo recibida de la empresa,27/02/2026,"629,58€",Salary,Income,629.58,2026-02-27,2026,2,2026-02
2,Actividad de ocio durante el viaje a París,25/02/2026,"51,00€","Leisure, Vacations",Expenses,51.00,2026-02-25,2026,2,2026-02
3,Entradas de cine y palomitas para ver una pelí...,22/02/2026,"2,80€",Leisure,Expenses,2.80,2026-02-22,2026,2,2026-02
4,Compra de bebidas y snacks en Eroski,21/02/2026,"24,80€",Food,Expenses,24.80,2026-02-21,2026,2,2026-02
5,Cesta de la compra básica en el el mercado cen...,21/02/2026,"21,60€",Food,Expenses,21.60,2026-02-21,2026,2,2026-02
6,Liquidación de recibo de Gas,21/02/2026,"6,50€",Invoice,Expenses,6.50,2026-02-21,2026,2,2026-02
7,Ronda de cervezas y refrescos con María en Vips,09/02/2026,"3,90€",Leisure,Expenses,3.90,2026-02-09,2026,2,2026-02
8,Ronda de cervezas y refrescos con María en Sta...,08/02/2026,"8,00€",Leisure,Expenses,8.00,2026-02-08,2026,2,2026-02
9,Ronda de cervezas y refrescos con Elena en Cub...,08/02/2026,"4,50€",Leisure,Expenses,4.50,2026-02-08,2026,2,2026-02


In [3]:
# Celda 3 - Analytics basico

# Separar ingresos y gastos
expenses = df[df["Type"] == "Expenses"]
income = df[df["Type"] == "Income"]

total_income = income["Amount_clean"].sum()
total_expenses = expenses["Amount_clean"].sum()
net_savings = total_income - total_expenses

print("=" * 50)
print("RESUMEN FINANCIERO GLOBAL")
print("=" * 50)
print(f"Total ingresos:  {total_income:>10,.2f} EUR")
print(f"Total gastos:    {total_expenses:>10,.2f} EUR")
print(f"Ahorro neto:     {net_savings:>10,.2f} EUR")
print(f"Tasa de ahorro:  {(net_savings/total_income*100) if total_income > 0 else 0:.1f}%")

# Breakdown por categoria
print(f"\n{'='*50}")
print("GASTOS POR CATEGORIA")
print(f"{'='*50}")
cat_breakdown = (
    expenses.groupby("Area")["Amount_clean"]
    .agg(["sum", "count", "mean"])
    .sort_values("sum", ascending=False)
)
cat_breakdown["pct"] = cat_breakdown["sum"] / total_expenses * 100
cat_breakdown.columns = ["Total (EUR)", "N Transacciones", "Media (EUR)", "% del Total"]
print(cat_breakdown.to_string(float_format="{:.2f}".format))

# Resumen mensual reciente (ultimos 6 meses)
print(f"\n{'='*50}")
print("RESUMEN MENSUAL (ultimos 6 meses)")
print(f"{'='*50}")
monthly = df.groupby(["YearMonth", "Type"])["Amount_clean"].sum().unstack(fill_value=0)
monthly = monthly.sort_index(ascending=False).head(6)
if "Income" not in monthly.columns:
    monthly["Income"] = 0
if "Expenses" not in monthly.columns:
    monthly["Expenses"] = 0
monthly["Ahorro"] = monthly["Income"] - monthly["Expenses"]
monthly["Tasa Ahorro %"] = (monthly["Ahorro"] / monthly["Income"] * 100).replace([np.inf, -np.inf], 0).fillna(0)
print(monthly.to_string(float_format="{:.2f}".format))

RESUMEN FINANCIERO GLOBAL
Total ingresos:   23,196.76 EUR
Total gastos:     15,346.11 EUR
Ahorro neto:       7,850.65 EUR
Tasa de ahorro:  33.8%

GASTOS POR CATEGORIA
                    Total (EUR)  N Transacciones  Media (EUR)  % del Total
Area                                                                      
Invoice                 5243.41              241        21.76        34.17
Leisure, Vacations      2636.58               17       155.09        17.18
Food                    2438.43              176        13.85        15.89
Investment              2393.31               71        33.71        15.60
Leisure                 2161.10              175        12.35        14.08
Invoice, Vacations       302.97               13        23.31         1.97
Food, Vacations          170.31               20         8.52         1.11

RESUMEN MENSUAL (ultimos 6 meses)
Type       Expenses  Income  Ahorro  Tasa Ahorro %
YearMonth                                         
2026-02      399.60  

In [4]:
# Celda 4 - Deteccion de gastos recurrentes y anomalias

# Gastos recurrentes (suscripciones)
print("=" * 50)
print("GASTOS RECURRENTES DETECTADOS")
print("=" * 50)

# Agrupar por descripcion similar y buscar patrones mensuales
recurring_keywords = ["Gimnasio", "Netflix", "Spotify", "Seguro", "Gas", "Internet", "Agua"]
for kw in recurring_keywords:
    matches = expenses[expenses["Description"].str.contains(kw, case=False, na=False)]
    if len(matches) >= 2:
        avg_amount = matches["Amount_clean"].mean()
        n_months = matches["YearMonth"].nunique()
        print(f"  {kw}: ~{avg_amount:.2f} EUR/mes ({len(matches)} pagos en {n_months} meses)")

# Anomalias: gastos > media + 1.5*std por categoria
print(f"\n{'='*50}")
print("ANOMALIAS DETECTADAS (gastos inusualmente altos)")
print(f"{'='*50}")

for area in expenses["Area"].unique():
    area_data = expenses[expenses["Area"] == area]["Amount_clean"]
    if len(area_data) < 5:
        continue
    mean_val = area_data.mean()
    std_val = area_data.std()
    threshold = mean_val + 1.5 * std_val
    anomalies = expenses[(expenses["Area"] == area) & (expenses["Amount_clean"] > threshold)]
    for _, row in anomalies.iterrows():
        print(f"  [{area}] {row['Date']}: {row['Amount_clean']:.2f} EUR - {row['Description'][:60]}")
        print(f"          (media: {mean_val:.2f} EUR, umbral: {threshold:.2f} EUR)")

GASTOS RECURRENTES DETECTADOS
  Gimnasio: ~11.13 EUR/mes (25 pagos en 20 meses)
  Netflix: ~26.19 EUR/mes (24 pagos en 18 meses)
  Spotify: ~9.32 EUR/mes (29 pagos en 24 meses)
  Seguro: ~26.58 EUR/mes (29 pagos en 22 meses)
  Gas: ~10.84 EUR/mes (84 pagos en 43 meses)
  Internet: ~14.35 EUR/mes (28 pagos en 24 meses)
  Agua: ~54.77 EUR/mes (39 pagos en 28 meses)

ANOMALIAS DETECTADAS (gastos inusualmente altos)
  [Leisure] 23/11/2025: 50.00 EUR - Ronda de cervezas y refrescos con Diego en Starbucks
          (media: 12.35 EUR, umbral: 39.88 EUR)
  [Leisure] 10/08/2025: 56.60 EUR - Ronda de cervezas y refrescos con Marta en Vips
          (media: 12.35 EUR, umbral: 39.88 EUR)
  [Leisure] 21/06/2025: 59.77 EUR - Merienda en La Tagliatella por la tarde - pago procesado
          (media: 12.35 EUR, umbral: 39.88 EUR)
  [Leisure] 07/09/2024: 203.58 EUR - Ronda de cervezas y refrescos con María en el bar de la esqu
          (media: 12.35 EUR, umbral: 39.88 EUR)
  [Leisure] 12/08/2024: 46.0

In [5]:
# Celda 5 - Construccion del prompt con datos reales

def build_user_profile(df):
    """Construye un resumen del perfil financiero del usuario."""
    expenses = df[df["Type"] == "Expenses"]
    income = df[df["Type"] == "Income"]
    total_inc = income["Amount_clean"].sum()
    total_exp = expenses["Amount_clean"].sum()
    
    # Breakdown por categoria
    cat_summary = (
        expenses.groupby("Area")["Amount_clean"]
        .sum()
        .sort_values(ascending=False)
    )
    cat_lines = [f"  - {area}: {amount:.2f} EUR ({amount/total_exp*100:.1f}%)" 
                 for area, amount in cat_summary.items()]
    
    # Ultimos 3 meses
    recent_months = df["YearMonth"].unique()[:3]
    recent = df[df["YearMonth"].isin(recent_months)]
    recent_exp = recent[recent["Type"] == "Expenses"]["Amount_clean"].sum()
    recent_inc = recent[recent["Type"] == "Income"]["Amount_clean"].sum()
    
    profile = (
        f"Periodo de datos: {df['Date_parsed'].min().strftime('%d/%m/%Y')} - {df['Date_parsed'].max().strftime('%d/%m/%Y')}\n"
        f"Total transacciones: {len(df)}\n"
        f"Ingresos totales: {total_inc:,.2f} EUR\n"
        f"Gastos totales: {total_exp:,.2f} EUR\n"
        f"Ahorro neto: {total_inc - total_exp:,.2f} EUR\n"
        f"Tasa de ahorro global: {((total_inc - total_exp)/total_inc*100) if total_inc > 0 else 0:.1f}%\n"
        f"\nDesglose de gastos por categoria:\n" + "\n".join(cat_lines) + "\n"
        f"\nUltimos 3 meses:\n"
        f"  Ingresos: {recent_inc:,.2f} EUR\n"
        f"  Gastos: {recent_exp:,.2f} EUR\n"
        f"  Ahorro: {recent_inc - recent_exp:,.2f} EUR"
    )
    return profile


def build_relevant_transactions(df, area=None, n=15):
    """Formatea transacciones relevantes para incluir en el prompt."""
    subset = df.copy()
    if area:
        subset = subset[subset["Area"].str.contains(area, case=False, na=False)]
    subset = subset.head(n)
    
    lines = []
    for _, row in subset.iterrows():
        tipo = "Gasto" if row["Type"] == "Expenses" else "Ingreso"
        lines.append(
            f"- {row['Date']} | {row['Description']} | {row['Amount_clean']:.2f} EUR | {row['Area']} | {tipo}"
        )
    return "\n".join(lines)


SYSTEM_PROMPT = """Eres un coach financiero personal experto y amable. Siempre respondes en espanol.

REGLAS ESTRICTAS:
- Basa tus respuestas EXCLUSIVAMENTE en los datos financieros reales del usuario proporcionados abajo.
- NUNCA inventes transacciones, cifras o datos que no aparezcan en el contexto.
- Cuando expliques conceptos financieros, usa ejemplos concretos de las transacciones del usuario.
- Se motivador y practico. Da consejos especificos y accionables.
- Formatea cantidades en EUR con dos decimales.
- Si no tienes datos suficientes para responder, dilo honestamente.

PERFIL FINANCIERO DEL USUARIO:
{profile}

TRANSACCIONES RECIENTES RELEVANTES:
{transactions}"""


# Construir el contexto
user_profile = build_user_profile(df)
recent_transactions = build_relevant_transactions(df, n=20)

system_message = SYSTEM_PROMPT.format(
    profile=user_profile,
    transactions=recent_transactions
)

print("System prompt construido:")
print(f"Longitud: {len(system_message)} caracteres")
print("\n" + "=" * 50)
print(system_message[:1500] + "\n...")

System prompt construido:
Longitud: 3034 caracteres

Eres un coach financiero personal experto y amable. Siempre respondes en espanol.

REGLAS ESTRICTAS:
- Basa tus respuestas EXCLUSIVAMENTE en los datos financieros reales del usuario proporcionados abajo.
- NUNCA inventes transacciones, cifras o datos que no aparezcan en el contexto.
- Cuando expliques conceptos financieros, usa ejemplos concretos de las transacciones del usuario.
- Se motivador y practico. Da consejos especificos y accionables.
- Formatea cantidades en EUR con dos decimales.
- Si no tienes datos suficientes para responder, dilo honestamente.

PERFIL FINANCIERO DEL USUARIO:
Periodo de datos: 01/09/2021 - 28/02/2026
Total transacciones: 885
Ingresos totales: 23,196.76 EUR
Gastos totales: 15,346.11 EUR
Ahorro neto: 7,850.65 EUR
Tasa de ahorro global: 33.8%

Desglose de gastos por categoria:
  - Invoice: 5243.41 EUR (34.2%)
  - Leisure, Vacations: 2636.58 EUR (17.2%)
  - Food: 2438.43 EUR (15.9%)
  - Investment: 2393.31 

In [6]:
# Celda 6 - Test con multiples modelos
# Comparar respuestas de distintos LLMs a las mismas preguntas

MODELS = {
    "Llama 3.3 70B": "llama-3.3-70b-versatile",
    "Llama 3.1 8B": "llama-3.1-8b-instant",
    "Mixtral 8x7B": "mixtral-8x7b-32768",
    "Gemma 2 9B": "gemma2-9b-it",
}

TEST_QUESTIONS = [
    "Dame consejos para ahorrar mas dinero basandote en mis gastos.",
    "Explicame que es el interes compuesto usando ejemplos de mis inversiones.",
    "Cuanto gasto al mes en ocio? Es demasiado?",
]


def ask_model(model_id, question, system_msg):
    """Envia una pregunta a un modelo via Groq y devuelve la respuesta."""
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": question},
            ],
            temperature=0.7,
            max_tokens=1024,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"ERROR: {e}"


# Ejecutar comparativa
results = {}
for q_idx, question in enumerate(TEST_QUESTIONS):
    print(f"\n{'#' * 70}")
    print(f"PREGUNTA {q_idx + 1}: {question}")
    print(f"{'#' * 70}")
    
    results[question] = {}
    for model_name, model_id in MODELS.items():
        print(f"\n--- {model_name} ({model_id}) ---")
        answer = ask_model(model_id, question, system_message)
        results[question][model_name] = answer
        # Mostrar primeros 500 chars para comparar rapidamente
        preview = answer[:500] + "..." if len(answer) > 500 else answer
        print(preview)


######################################################################
PREGUNTA 1: Dame consejos para ahorrar mas dinero basandote en mis gastos.
######################################################################

--- Llama 3.3 70B (llama-3.3-70b-versatile) ---
Basándome en tus gastos, puedo ver que tienes un buen equilibrio entre ingresos y gastos, con una tasa de ahorro global del 33,8%. Sin embargo, hay algunas áreas en las que podrías reducir tus gastos para ahorrar aún más.

**Gastos en ocio y vacaciones**: Noté que has gastado un total de 4.797,68 EUR (2636,58 EUR en Leisure, Vacations y 2161,10 EUR en Leisure) en ocio y vacaciones, lo que representa el 31,2% de tus gastos totales. Podrías considerar reducir un poco estos gastos, como cancelar su...

--- Llama 3.1 8B (llama-3.1-8b-instant) ---
Basándome en tus gastos, te proporciono algunos consejos para ahorrar más dinero:

1. **Reducir los gastos en alimentos y bebidas**: Tus gastos en comida y bebidas son significativos (

In [7]:
# Celda 7 - Ver respuestas completas del modelo que mas te guste
# Cambia el nombre del modelo para ver su respuesta completa

SELECTED_MODEL = "Llama 3.3 70B"  # Cambiar aqui

for question, model_answers in results.items():
    print(f"\n{'=' * 70}")
    print(f"PREGUNTA: {question}")
    print(f"{'=' * 70}")
    print(f"\nRESPUESTA ({SELECTED_MODEL}):\n")
    print(model_answers.get(SELECTED_MODEL, "Modelo no encontrado"))


PREGUNTA: Dame consejos para ahorrar mas dinero basandote en mis gastos.

RESPUESTA (Llama 3.3 70B):

Basándome en tus gastos, puedo ver que tienes un buen equilibrio entre ingresos y gastos, con una tasa de ahorro global del 33,8%. Sin embargo, hay algunas áreas en las que podrías reducir tus gastos para ahorrar aún más.

**Gastos en ocio y vacaciones**: Noté que has gastado un total de 4.797,68 EUR (2636,58 EUR en Leisure, Vacations y 2161,10 EUR en Leisure) en ocio y vacaciones, lo que representa el 31,2% de tus gastos totales. Podrías considerar reducir un poco estos gastos, como cancelar suscripciones a servicios de entretenimiento que no utilices con frecuencia o buscar alternativas más económicas para tus vacaciones.

**Gastos en comida**: Tus gastos en comida ascienden a 2.438,43 EUR, lo que representa el 15,9% de tus gastos totales. Podrías considerar planificar tus comidas y hacer compras de manera más eficiente, como comprar productos en oferta o utilizar cupones de descuen

In [8]:
# Celda 8 - Test conversacional multi-turno
# Simula una conversacion de varios turnos con el coach

CHAT_MODEL = "llama-3.3-70b-versatile"  # Cambiar si prefieres otro

conversation_history = [
    {"role": "system", "content": system_message},
]

# Simulacion de conversacion
test_conversation = [
    "Hola! Que puedes hacer por mi?",
    "Cuanto gasto al mes de media? En que categorias gasto mas?",
    "Y como podria reducir esos gastos? Dame consejos concretos.",
    "Que porcentaje de mis ingresos deberia estar ahorrando? Como lo estoy haciendo?",
]

for user_msg in test_conversation:
    print(f"\n{'=' * 60}")
    print(f"USUARIO: {user_msg}")
    print(f"{'=' * 60}")
    
    conversation_history.append({"role": "user", "content": user_msg})
    
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=conversation_history,
        temperature=0.7,
        max_tokens=1024,
    )
    
    assistant_msg = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_msg})
    
    print(f"\nCOACH:\n{assistant_msg}")
    print(f"\n[Tokens usados: {response.usage.total_tokens}]")

print(f"\n\nTotal turnos: {len(conversation_history) - 1}")
print(f"Mensajes en historial: {len(conversation_history)}")


USUARIO: Hola! Que puedes hacer por mi?

COACH:
¡Hola! Me alegra conocerte. Como coach financiero personal, puedo ayudarte a analizar tus finanzas y proporcionarte consejos prácticos para mejorar tu situación económica.

Con los datos que me has proporcionado, puedo ver que tienes un historial de transacciones de 885 operaciones, con un total de ingresos de 23.196,76 EUR y gastos de 15.346,11 EUR, lo que te deja un ahorro neto de 7.850,65 EUR.

Puedo ayudarte a:

* Analizar tus gastos y identificar áreas donde puedes reducir costos
* Establecer metas financieras y crear un plan para alcanzarlas
* Proporcionarte consejos para aumentar tus ingresos y reducir tus gastos
* Evaluar tus inversiones y proporcionarte recomendaciones para optimizar tus finanzas

¿Qué te gustaría trabajar en primer lugar? ¿Tienes alguna meta financiera específica en mente o algún área de preocupación en particular?

[Tokens usados: 1336]

USUARIO: Cuanto gasto al mes de media? En que categorias gasto mas?

COAC

In [9]:
# Celda 9 - Test con contexto especifico por intent
# Probar como cambia la respuesta al dar transacciones especificas de una categoria

# Ejemplo: pregunta sobre inversiones con contexto de inversiones
investment_transactions = build_relevant_transactions(df, area="Investment", n=15)

investment_prompt = SYSTEM_PROMPT.format(
    profile=user_profile,
    transactions=investment_transactions
)

print("Transacciones de inversion incluidas en el prompt:")
print(investment_transactions)
print(f"\n{'=' * 60}")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": investment_prompt},
        {"role": "user", "content": "Explicame que es la diversificacion de inversiones usando mis propios datos como ejemplo."},
    ],
    temperature=0.7,
    max_tokens=1024,
)

print(f"\nRESPUESTA DEL COACH:\n")
print(response.choices[0].message.content)

Transacciones de inversion incluidas en el prompt:
- 07/02/2026 | Operación de compra de activos en letras del tesoro | 20.00 EUR | Investment | Gasto
- 01/02/2026 | Aportación periódica al fondo de inversión letras del tesoro | 180.00 EUR | Investment | Gasto
- 18/01/2026 | Aportación periódica al fondo de inversión acciones de Apple | 20.00 EUR | Investment | Gasto
- 01/01/2026 | Inversión mensual en acciones de Apple | 180.00 EUR | Investment | Gasto
- 4/12/2025 | Aportación periódica al fondo de inversión acciones de Apple. | 20.00 EUR | Investment | Gasto
- 4/12/2025 | Operación de compra de activos en S&P 500 | 180.00 EUR | Investment | Gasto
- 12/11/2025 | Operación de compra de activos en S&P 500. | 100.00 EUR | Investment | Gasto
- 1/11/2025 | Aportación periódica al fondo de inversión acciones de Apple - pago procesado | 50.00 EUR | Investment | Gasto
- 1/11/2025 | Operación de compra de activos en acciones de Apple | 50.00 EUR | Investment | Gasto
- 1/10/2025 | Compra de pla

## Conclusiones del prototipo

Tras ejecutar este notebook, evaluar:

1. **Cual modelo da mejor calidad de respuesta en espanol?**
2. **Las respuestas usan datos reales o inventan cifras?**
3. **El contexto especifico por categoria mejora las respuestas?**
4. **La conversacion multi-turno mantiene coherencia?**

### Siguiente paso
Migrar la logica validada aqui a los modulos de `src/` para produccion.